<a href="https://colab.research.google.com/github/XAMANEKR/compaplica_mita/blob/main/Hands_On_Pipeline_Completo_(Soluci%C3%B3n).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PIPELINE COMPLETO: DE DATOS A MODELO DESPLEGADO**

Este Colab conecta en un solo flujo lo visto en los tres Temas anteriores (inferencia básica, RAG y fine-tuning con LoRa) para construir un asistente que recupera contexto propio y responde con un modelo ya ajustado a un tono específico. Es la base directa para el reto del Hackathon 1.

**Nota de alcance:** "Desplegar" aquí significa guardar el modelo ajustado de forma reutilizable y envolverlo en una función lista para usar, no se levanta un servidor ni un endpoint público.

## **CONFIGURACIÓN DEL ENTORNO**

### **INSTALAR LIBRERÍAS**

En este Tema **no necesitas API key ni token**. El modelo base es una copia pública de **Llama 3.2 de 1B Instruct** y el modelo de embeddings también se descarga del Hub público de Hugging Face; si Colab muestra un aviso de *"unauthenticated requests to the HF Hub"* es normal, solo indica que descargas sin cuenta. El modelo grande de la última etapa corre con Ollama.

In [ ]:
# Instalar librerías (no se necesita token ni inicio de sesión)

!pip install transformers peft accelerate trl sentence-transformers --quiet

import torch
import logging
logging.getLogger("huggingface_hub.utils._http").setLevel(logging.ERROR)  # oculta el aviso de descargas sin token

from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

## **PASO 1: LA BASE DE CONOCIMIENTO (RAG)**

Indexamos una base de conocimiento propia con `sentence-transformers` para poder recuperar el fragmento más relevante antes de responder, esto es lo que evita que el asistente alucine sobre políticas que no conoce.

In [ ]:
# Generar los embeddings de la base de conocimiento (política de devoluciones)

from sentence_transformers import SentenceTransformer
import numpy as np

modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "Los libros se prestan por un máximo de 15 días, con posibilidad de renovar una vez si nadie más lo ha solicitado.",
    "Las devoluciones tardías generan una multa de $10 por día de retraso, hasta un "
    "máximo de $150 por libro.",
    "Los libros de la sección de reserva (material de examen) solo se consultan dentro de la biblioteca, no se prestan a domicilio."
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generados: (3, 384)


In [ ]:
# Definir la función de recuperación (RAG)

def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

buscar_fragmento("¿Puedo llevarme a casa un libro de la sección de reserva?")

'Los libros se prestan por un máximo de 15 días, con posibilidad de renovar una vez si nadie más lo ha solicitado.'

## **PASO 2: AJUSTAR EL TONO DEL MODELO (FINE-TUNING CON LoRA)**

Cargamos un modelo ligero de Llama (**Llama 3.2 de 1B Instruct**, en una copia pública alojada por Unsloth; la versión oficial `meta-llama/Llama-3.2-1B-Instruct` requiere cuenta y aceptar la licencia de Meta) y lo ajustamos con LoRA para que responda siempre en el mismo tono breve y directo, la configuración de esta celda ya fue validada en el tema 3.

In [ ]:
# Cargar el modelo base de Llama y su tokenizer

modelo_base = "unsloth/Llama-3.2-1B-Instruct"
# variante oficial de Meta (requiere cuenta de Hugging Face, token y aceptar la licencia): "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(modelo_base)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # Llama 3.2 no define token de relleno
modelo = AutoModelForCausalLM.from_pretrained(modelo_base, dtype=torch.float16, device_map="auto")
print("Modelo base cargado:", modelo_base)

config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Modelo base cargado: unsloth/Llama-3.2-1B-Instruct


In [ ]:
def generar_respuesta(modelo_a_usar, pregunta, max_new_tokens=60):
    mensajes = [{"role": "user", "content": pregunta}]
    prompt_formateado = tokenizer.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
    entrada = tokenizer(prompt_formateado, return_tensors="pt").to(modelo_a_usar.device)
    salida = modelo_a_usar.generate(
        **entrada, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id, no_repeat_ngram_size=3,
    )
    tokens_nuevos = salida[0][entrada["input_ids"].shape[1]:]
    return tokenizer.decode(tokens_nuevos, skip_special_tokens=True).strip()

prompt_prueba = "¿Puedo renovar un préstamo?"

In [ ]:
# Definir el dataset de ejemplos y convertirlo en Dataset

from datasets import Dataset

def formatear_ejemplo(pregunta, respuesta):
    mensajes = [
        {"role": "user", "content": pregunta},
        {"role": "assistant", "content": respuesta}
    ]
    return tokenizer.apply_chat_template(mensajes, tokenize=False)

pares = [
    ("¿Puedo renovar un préstamo?", "Sí, puedes renovarlo una vez si nadie más lo ha solicitado."),
    ("¿Hasta cuándo puedo tener un libro prestado?", "El préstamo estándar es de 15 días."),
    ("¿Necesito credencial para entrar a la biblioteca?", "Sí, es necesario mostrar tu credencial vigente en la entrada."),
    ("¿Puedo reservar una sala de estudio?", "Sí, puedes reservarla hasta con 2 días de anticipación desde el portal."),
    ("¿Hay wifi disponible dentro de la biblioteca?", "Sí, la red 'Biblioteca-Invitados' está disponible en todas las salas."),
]

ejemplos = [{"texto": formatear_ejemplo(p, r)} for p, r in pares]
dataset = Dataset.from_list(ejemplos)
dataset

Dataset({
    features: ['texto'],
    num_rows: 5
})

In [ ]:
# Probar el modelo base con el prompt de prueba antes de ajustarlo

respuesta_base = generar_respuesta(modelo, prompt_prueba)
print(respuesta_base)

Sí, puedes renovar tu préstado. La renovación de un présto depende de varios factores, como el tiempo que ha transcurrido desde el inicio del préstambo, el monto del présto, y las condiciones del préstito.


In [ ]:
# Configurar LoRA (semilla fija para un resultado reproducible en la grabación)

!pip uninstall -y torchao > /dev/null 2>&1 # evita un error de peft si Colab trae un torchao viejo

from peft import LoraConfig, get_peft_model
from transformers import set_seed
set_seed(42)

config_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.0,
    task_type="CAUSAL_LM"
)

modelo_lora = get_peft_model(modelo, config_lora)
modelo_lora.print_trainable_parameters()

trainable params: 851,968 || all params: 1,236,666,368 || trainable%: 0.0689


In [ ]:
# Entrenar con LoRA

from trl import SFTTrainer, SFTConfig

config_entrenamiento = SFTConfig(
    output_dir="/content/resultados_pipeline",
    num_train_epochs=30,
    per_device_train_batch_size=5,
    learning_rate=2e-4,
    logging_steps=1,
    dataset_text_field="texto",
    max_length=128,
    report_to="none",
    fp16=True,   # la T4 no soporta bf16 (TRL lo activa por defecto y falla)
    bf16=False,
)

trainer = SFTTrainer(
    model=modelo_lora,
    train_dataset=dataset,
    args=config_entrenamiento,
)

resultado_entrenamiento = trainer.train()
# Dejar el modelo listo para generar texto: TRL lo deja en modo entrenamiento y sin caché
modelo_lora.eval()
modelo_lora.config.use_cache = True
if hasattr(modelo_lora, "gradient_checkpointing_disable"):
    modelo_lora.gradient_checkpointing_disable()
perdida_inicial = trainer.state.log_history[0]['loss']
perdida_final = trainer.state.log_history[-2]['loss'] # último paso del entrenamiento
#perdida_final = resultado_entrenamiento.training_loss # promedio del entrenamiento
print(f"Pérdida al inicio del entrenamiento: {perdida_inicial:.2f}")
print(f"Pérdida final del entrenamiento: {perdida_final:.2f}")
print(f"Reducción: {(1 - perdida_final/perdida_inicial) * 100:.0f}%")

Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

{'loss': '4.991', 'grad_norm': '5.32', 'learning_rate': '0.0002', 'entropy': '1.975', 'mean_token_accuracy': '0.4092', 'num_tokens': '330', 'epoch': '1'}
{'loss': '4.78', 'grad_norm': '4.676', 'learning_rate': '0.0001933', 'entropy': '2.053', 'mean_token_accuracy': '0.4092', 'num_tokens': '660', 'epoch': '2'}
{'loss': '4.546', 'grad_norm': '4.456', 'learning_rate': '0.0001867', 'entropy': '2.132', 'mean_token_accuracy': '0.44', 'num_tokens': '990', 'epoch': '3'}
{'loss': '4.325', 'grad_norm': '4.411', 'learning_rate': '0.00018', 'entropy': '2.188', 'mean_token_accuracy': '0.4708', 'num_tokens': '1320', 'epoch': '4'}
{'loss': '4.115', 'grad_norm': '4.306', 'learning_rate': '0.0001733', 'entropy': '2.23', 'mean_token_accuracy': '0.4862', 'num_tokens': '1650', 'epoch': '5'}
{'loss': '3.918', 'grad_norm': '4.148', 'learning_rate': '0.0001667', 'entropy': '2.254', 'mean_token_accuracy': '0.4985', 'num_tokens': '1980', 'epoch': '6'}
{'loss': '3.737', 'grad_norm': '4.161', 'learning_rate': '0

## **PASO 3: DESPLEGAR EL MODELO AJUSTADO**

"Desplegar" en este contexto significa guardar el adaptador LoRA de forma reutilizable, no levantar un servidor. Con `save_pretrained` queda listo para volver a cargarlo en cualquier notebook sin repetir el entrenamiento; `push_to_hub` es opcional si quieres tenerlo disponible en tu cuenta de Hugging Face.

In [ ]:
# Guardar el adaptador LoRA localmente

modelo_lora.save_pretrained("/content/modelo_ajustado_lora")
print("Adaptador LoRA guardado en /content/modelo_ajustado_lora")

# Opcional: subir el adaptador a tu cuenta de Hugging Face para reutilizarlo fuera de esta sesión
# modelo_lora.push_to_hub("tu-usuario/llama-atencion-clientes-lora")

Adaptador LoRA guardado en /content/modelo_ajustado_lora


## **PASO 4: EL PIPELINE COMPLETO — RAG + MODELO AJUSTADO**

Con la base de conocimiento indexada y el modelo ya ajustado, conectamos ambas piezas en una sola función: recupera el fragmento relevante, se lo entrega al modelo ajustado junto con la pregunta, y genera la respuesta final. Este es el mismo patrón que se espera construir en el Hackathon 1.

***Nota:** El modelo de 1B aprendió un tono específico con solo 5 ejemplos, suficiente para demostrar que LoRA funciona, pero un modelo tan pequeño y con tan pocos ejemplos no es confiable para seguir una instrucción nueva con contexto inyectado, como la que pide este pipeline con RAG.*

*Por lo que, un modelo real desplegado sí tendría esa capacidad, usando aquí Llama 3.1 8B (Ollama) en lugar del modelo ajustado con LoRA como muestra de ese comportamiento.*

### **CONFIGURAR OLLAMA (MODELO GRANDE)**

Para la última etapa instalamos Ollama (igual que en el Tema 1) y descargamos **Llama 3.1 de 8B**. Si tu sesión de Colab es nueva, esto descarga ~5 GB.

**IMPORTANTE:** la sesión de Colab es temporal. Si se reinicia o se desconecta, hay que volver a ejecutar las celdas de instalación, servidor y descarga del modelo.

In [ ]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)

!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
print("Ollama instalado.")

Ollama instalado.


In [ ]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda

import subprocess
import time
import requests

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Servidor de Ollama listo.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

!ollama --version

Servidor de Ollama listo.
ollama version is 0.40.12


In [ ]:
# Descargar el modelo Llama (la primera vez tarda unos minutos)

MODELO = "llama3.1:8b"

!ollama pull {MODELO}
!ollama list


NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    


In [ ]:
# Instalar el cliente de Ollama y conectarlo al servidor local

!pip install ollama -q

import ollama

client = ollama.Client(host="http://localhost:11434")
print("Cliente de Ollama inicializado correctamente.")

Cliente de Ollama inicializado correctamente.


In [ ]:
# Función del asistente: RAG (recuperación) + modelo grande (generación con Llama 3.1 8B en Ollama)

import gc
gc.collect()
torch.cuda.empty_cache()  # libera memoria de la GPU para el modelo grande

def asistente(pregunta):
    fragmento = buscar_fragmento(pregunta)
    prompt = f"""Responde la pregunta del alumno usando SOLO el siguiente reglamento de la biblioteca. Si el reglamento no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

    response = client.chat(
        model=MODELO,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.message.content

for pregunta in [
    "¿Puedo llevarme a casa un libro de la sección de reserva (material de examen)?",
    "¿Cuánto es la multa por entregar un libro tarde?",
    "¿Puedo renovar el préstamo de un libro?",
]:
    print(f"Pregunta: {pregunta}")
    print(f"Respuesta: {asistente(pregunta)}\n")

Pregunta: ¿Puedo llevarme a casa un libro de la sección de reserva (material de examen)?
Respuesta: No, no puedes llevarte a casa un libro de la sección de reserva.

Pregunta: ¿Cuánto es la multa por entregar un libro tarde?
Respuesta: La multa por devolver un libro tarde es $10 por día de retraso, hasta un máximo de $150 por libro.

Pregunta: ¿Puedo renovar el préstamo de un libro?
Respuesta: Sí.



### **COMPARACIÓN: ¿Y EL MODELO AJUSTADO?**

Para ver qué aporta cada pieza, hacemos las mismas preguntas con el mismo RAG y el mismo prompt, pero generando la respuesta con el modelo de 1B ajustado con LoRA. Observa si respeta el reglamento recuperado o si solo imita el tono breve que aprendió.

In [ ]:
# Comparar el asistente con Llama 3.1 8B (Ollama) contra el modelo ajustado con LoRA (mismo RAG, mismo prompt)

def asistente_ajustado(pregunta):
    fragmento = buscar_fragmento(pregunta)
    prompt = f"""Responde la pregunta del alumno usando SOLO el siguiente reglamento de la biblioteca. Si el reglamento no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""
    return generar_respuesta(modelo_lora, prompt)

for pregunta in [
    "¿Puedo llevarme a casa un libro de la sección de reserva (material de examen)?",
    "¿Cuánto es la multa por entregar un libro tarde?",
    "¿Puedo renovar el préstamo de un libro?",
]:
    print(f"Pregunta: {pregunta}")
    print(f"Fragmento recuperado: {buscar_fragmento(pregunta)}")
    print(f"  Llama 3.1 8B (Ollama): {asistente(pregunta)}")
    print(f"  Llama 3.2 1B + LoRA:   {asistente_ajustado(pregunta)}\n")

Pregunta: ¿Puedo llevarme a casa un libro de la sección de reserva (material de examen)?
Fragmento recuperado: Los libros de la sección de reserva (material de examen) solo se consultan dentro de la biblioteca, no se prestan a domicilio.
  Llama 3.1 8B (Ollama): No, según la política de la biblioteca, los libros de la sección de reserva (material de examen) no se prestan a domicilio.
  Llama 3.2 1B + LoRA:   No

Pregunta: ¿Cuánto es la multa por entregar un libro tarde?
Fragmento recuperado: Las devoluciones tardías generan una multa de $10 por día de retraso, hasta un máximo de $150 por libro.
  Llama 3.1 8B (Ollama): La multa por entregar un libro tarde es de $10 por día de retraso, hasta un máximo de $150 por libro.
  Llama 3.2 1B + LoRA:   La respuesta es de $50.

Pregunta: ¿Puedo renovar el préstamo de un libro?
Fragmento recuperado: Los libros se prestan por un máximo de 15 días, con posibilidad de renovar una vez si nadie más lo ha solicitado.
  Llama 3.1 8B (Ollama): Sí, puedes